<a href="https://colab.research.google.com/github/esmondyu/AAI2026/blob/dev/Prompt_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# Colab 1: Prompt Chaining - Customer Support AI (No external API required)
# This simulates prompt chaining using deterministic Python logic + "prompt artifacts" for submission.

from dataclasses import dataclass
from typing import Dict, Any, List
import re
from collections import Counter

@dataclass
class TriageResult:
    issue_type: str
    urgency: str
    key_details: List[str]
    missing_info: List[str]
    suggested_next_step: str

def step_a_triage(customer_message: str) -> TriageResult:
    msg = customer_message.lower()

    if any(k in msg for k in ["refund", "charge", "charged", "billing", "invoice", "payment"]):
        issue_type = "billing"
    elif any(k in msg for k in ["login", "password", "locked", "account", "reset"]):
        issue_type = "account"
    elif any(k in msg for k in ["not working", "error", "bug", "crash", "issue", "broken"]):
        issue_type = "technical"
    elif any(k in msg for k in ["shipping", "delivery", "tracking", "arrive", "package"]):
        issue_type = "shipping"
    else:
        issue_type = "other"

    urgency = "high" if any(k in msg for k in ["urgent", "asap", "immediately", "can't", "cannot access"]) else "medium"

    key_details = []
    if re.search(r"\border\b\s*#?\s*(\w+)", msg):
        key_details.append("Order number mentioned")
    if re.search(r"\$\s*\d+", msg) or re.search(r"\d+\s*dollars", msg):
        key_details.append("Dollar amount mentioned")
    if "yesterday" in msg or "today" in msg:
        key_details.append("Timing mentioned")

    missing_info = []
    if issue_type in ["billing", "shipping"] and "order" not in msg:
        missing_info.append("Order number")
    if issue_type == "billing" and not ("$" in msg or "dollars" in msg):
        missing_info.append("Charge amount")
    if issue_type == "technical":
        missing_info.append("Device/OS and exact error message")

    suggested_next_step = "Ask one clarifying question to proceed."

    return TriageResult(issue_type, urgency, key_details, missing_info, suggested_next_step)

def step_b_followup_question(triage: TriageResult) -> str:
    if triage.missing_info:
        return f"Quick question: could you share your {triage.missing_info[0]} so I can help?"
    if triage.issue_type == "technical":
        return "Quick question: what device/OS are you using and what exact error message do you see?"
    return "Quick question: can you confirm the email on the account so I can locate your details?"

def step_c_final_response(original_message: str, triage: TriageResult, customer_answer: str) -> str:
    apology = "Sorry about that — I can help." if triage.issue_type in ["billing", "technical", "shipping", "account"] else "Thanks for reaching out — I can help."
    steps = [
        "I’m going to locate your details based on what you shared.",
        "Then I’ll confirm what happened and the next best fix.",
        "If needed, I’ll escalate this to our specialist team."
    ]
    return (
        f"{apology}\n\n"
        f"**What I can do now:**\n"
        f"- Issue type: {triage.issue_type} (urgency: {triage.urgency})\n"
        f"- Your info: {customer_answer}\n\n"
        f"**Next steps:**\n"
        + "\n".join([f"{i+1}. {s}" for i, s in enumerate(steps)]) +
        "\n\nIf you notice any new details (screenshots, exact timestamps, order #), send them and I’ll move faster."
    )

# --- Demo run (Successful Output) ---
customer_message = "I was charged $49 yesterday but my subscription still says inactive. This is urgent."
triage = step_a_triage(customer_message)
followup = step_b_followup_question(triage)
customer_answer = "The email is user@example.com and the charge was $49 on my card ending 1234."
final = step_c_final_response(customer_message, triage, customer_answer)

print("STEP A (Triage):", triage)
print("\nSTEP B (Follow-up Question):", followup)
print("\nSTEP C (Final Response):\n", final)


STEP A (Triage): TriageResult(issue_type='billing', urgency='high', key_details=['Dollar amount mentioned', 'Timing mentioned'], missing_info=['Order number'], suggested_next_step='Ask one clarifying question to proceed.')

STEP B (Follow-up Question): Quick question: could you share your Order number so I can help?

STEP C (Final Response):
 Sorry about that — I can help.

**What I can do now:**
- Issue type: billing (urgency: high)
- Your info: The email is user@example.com and the charge was $49 on my card ending 1234.

**Next steps:**
1. I’m going to locate your details based on what you shared.
2. Then I’ll confirm what happened and the next best fix.
3. If needed, I’ll escalate this to our specialist team.

If you notice any new details (screenshots, exact timestamps, order #), send them and I’ll move faster.
